In [1]:
from pathlib import Path
import pickle

import pandas as pd
import numpy as np
import inspect
import xarray as xr

import matplotlib.pyplot as plt
import torch
from neuralhydrology.evaluation import metrics, get_tester
from neuralhydrology.utils.config import Config
import neuralhydrology.modelzoo
from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.evaluation.prob_metrics import run_probabilistic_evaluation

In [ ]:
# This block ensures the code inside only runs when the script is executed directly
if __name__ == '__main__':

    # by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal
    # support
    if torch.cuda.is_available() or torch.backends.mps.is_available():
        start_run(config_file=Path("1_basin.yml"))

    # fall back to CPU-only mode
    else:
        start_run(config_file=Path("1_basin.yml"), gpu=-1)

2025-11-13 06:05:15,975: Logging to d:\github\neuralhydrology\neuralhydrology\own\operational_forecast\runs\test_run_1311_060515\output.log initialized.
2025-11-13 06:05:15,975: ### Folder structure created at d:\github\neuralhydrology\neuralhydrology\own\operational_forecast\runs\test_run_1311_060515
2025-11-13 06:05:15,975: ### Run configurations for test_run
2025-11-13 06:05:15,975: experiment_name: test_run
2025-11-13 06:05:15,975: use_frequencies: ['2W-MON', '4W-MON']
2025-11-13 06:05:15,975: train_basin_file: 1_basin.txt
2025-11-13 06:05:15,975: validation_basin_file: 1_basin.txt
2025-11-13 06:05:15,975: test_basin_file: 1_basin.txt
2025-11-13 06:05:15,975: train_start_date: 1999-09-27 00:00:00
2025-11-13 06:05:15,975: train_end_date: 2008-08-31 00:00:00
2025-11-13 06:05:15,975: validation_start_date: 1996-09-30 00:00:00
2025-11-13 06:05:15,975: validation_end_date: 1999-09-12 00:00:00
2025-11-13 06:05:15,975: test_start_date: 1989-10-02 00:00:00
2025-11-13 06:05:15,975: test_end

2025-11-13 13:28:23,354: Exception in callback BaseSelectorEventLoop._read_from_self()
handle: <Handle BaseSelectorEventLoop._read_from_self()>
Traceback (most recent call last):
  File "c:\Users\Workstation\anaconda3\envs\neuralhydrology\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\Workstation\anaconda3\envs\neuralhydrology\lib\asyncio\selector_events.py", line 115, in _read_from_self
    data = self._ssock.recv(4096)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
2025-11-13 13:28:23,354: Exception in callback BaseSelectorEventLoop._read_from_self()
handle: <Handle BaseSelectorEventLoop._read_from_self()>
Traceback (most recent call last):
  File "c:\Users\Workstation\anaconda3\envs\neuralhydrology\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\Workstation\anaconda3\envs\neuralhydrology\lib\asyncio\selector_

In [ ]:
run_dir = Path("runs/test_run_1311_060515")
eval_run(run_dir=run_dir, period="test")

In [ ]:
with open(run_dir / "test" / "model_epoch020" / "test_results.p", "rb") as fp:
    results = pickle.load(fp)
    
results.keys()

In [ ]:
# Load validation results from the last epoch
df = pd.read_csv(run_dir / "validation" / "model_epoch020" / "validation_metrics.csv", dtype={'basin': str})
df = df.set_index('basin')

# Compute the median NSE from all basins, where discharge observations are available for that period
print(f"Median NSE of the validation period {df['NSE_1h'].median():.3f}")

In [ ]:
basin_id = '01031500' # Your basin ID
freq = '1D'           # Your frequency key
target_variable = 'QObs(mm/h)' # Your target variable name

# --- 1. Extract Observations ---
# Get the original xarray DataArray for observations. Its shape is (2557, 24)
qobs_original_xr = results[basin_id][freq]['xr'][f'{target_variable}_obs']

# --- 2. Extract Simulated Output with Samples ---
# Get the original xarray DataArray for simulated data.
qsim_with_samples_original_xr = results[basin_id][freq]['xr'][f'{target_variable}_sim']

# --- Prepare data for plotting (flattening to single 1D series) ---

# Get the daily dates coordinate from the original observation DataArray.
# Assuming 'date' is the dimension name for days, and the other for hours (e.g., 'time_step').
# You might need to adjust 'date' and 'time_step' to your actual dimension names.
daily_dates_coord = qobs_original_xr['date'].values
num_hours_per_day = qobs_original_xr.sizes[qobs_original_xr.dims[1]] # Assuming the second dim is hourly

# Construct the full hourly datetime index for the X-axis for plotting
full_hourly_dates = []
for d in daily_dates_coord:
    for h in range(num_hours_per_day):
        full_hourly_dates.append(pd.to_datetime(d) + pd.Timedelta(hours=h))
full_hourly_dates = np.array(full_hourly_dates) # This will be (2557 * 24,) = (61368,)

# Flatten the observed discharge data into a single 1D NumPy array
qobs_flat_np = qobs_original_xr.values.flatten() # (2557, 24) -> (61368,)

# --- 3. Calculate Quantiles from Samples and flatten ---
# Convert the xarray DataArray to a NumPy array
qsim_data_np = qsim_with_samples_original_xr.values # e.g., (2557, 24, 100)

# Calculate percentiles and mean across the 'samples' dimension (axis=-1)
# Then, flatten each result to a single 1D NumPy array for plotting
median_sim_flat_np = np.nanpercentile(qsim_data_np, 50, axis=-1).flatten() # (2557, 24) -> (61368,)
q25_flat_np = np.nanpercentile(qsim_data_np, 25, axis=-1).flatten()
q75_flat_np = np.nanpercentile(qsim_data_np, 75, axis=-1).flatten()
lower_bound_90ci_flat_np = np.nanpercentile(qsim_data_np, 5, axis=-1).flatten()
upper_bound_90ci_flat_np = np.nanpercentile(qsim_data_np, 95, axis=-1).flatten()
mean_sim_flat_np = np.nanmean(qsim_data_np, axis=-1).flatten()

# --- Create new 1D xarray DataArrays for consistent plotting ---
# These DataArrays will now all be 1D, with 'date' as their single dimension.
# This ensures compatibility for plotting with Matplotlib.
qobs = xr.DataArray(qobs_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
median_sim = xr.DataArray(median_sim_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
q25 = xr.DataArray(q25_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
q75 = xr.DataArray(q75_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
lower_bound_90ci = xr.DataArray(lower_bound_90ci_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
upper_bound_90ci = xr.DataArray(upper_bound_90ci_flat_np, coords={'date': full_hourly_dates}, dims=['date'])
mean_sim = xr.DataArray(mean_sim_flat_np, coords={'date': full_hourly_dates}, dims=['date'])

# --- 4. Plotting with Matplotlib for Uncertainty ---
fig, ax = plt.subplots(figsize=(16, 10))

# Plot observations (using the new 1D qobs DataArray)
ax.plot(qobs['date'].values, qobs.values, color='black', label='Observations', linewidth=1)

# Plot median prediction
ax.plot(median_sim['date'].values, median_sim.values, color='red', linestyle='--', label='CMAL Median Prediction', linewidth=1)
ax.plot(mean_sim['date'].values, mean_sim.values, color='blue', linestyle=':', label='CMAL Mean Prediction', linewidth=1)

# Plot uncertainty bands using fill_between
# 90% Confidence Interval (between 5th and 95th percentiles)
ax.fill_between(
    lower_bound_90ci['date'].values, # X-axis values (full hourly dates)
    lower_bound_90ci.values,         # Y1 values (flattened to 1D)
    upper_bound_90ci.values,         # Y2 values (flattened to 1D)
    color='red',
    alpha=0.2, # Lighter transparency for the wider band
    label='CMAL 90% Uncertainty Interval'
)

# 50% Confidence Interval (between 25th and 75th percentiles)
ax.fill_between(
    q25['date'].values,
    q25.values,
    q75.values,
    color='red',
    alpha=0.4, # Darker transparency for the narrower band
    label='CMAL 50% Uncertainty Interval'
)

ax.set_ylabel(f"{target_variable} (mm/d)")
ax.set_xlabel("Date")
ax.set_title(f"Test period - CMAL Prediction with Uncertainty for Basin {basin_id}")
ax.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate all metrics for the median prediction
values = metrics.calculate_all_metrics(qobs, mean_sim)

# Print the calculated metrics
print("\nMetrics for CMAL Median Prediction:")
for key, val in values.items():
    print(f"{key}: {val:.3f}")

In [ ]:
# Select a random basins from the lower 50% of the NSE distribution
basin = df.loc[df["NSE_1h"] < df["NSE_1h"].median()].sample(n=1).index[0]

print(f"Selected basin: {basin} with an NSE of {df.loc[df.index == basin, 'NSE_1h'].values[0]:.3f}")

In [ ]:
# Add the path to the pre-trained model to the finetune config
with open("finetune.yml", "a") as fp:
    fp.write(f"\nbase_run_dir: {run_dir.absolute()}")
    
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [ ]:
finetune(Path("finetune.yml"))

In [ ]:
finetune_dir = Path("runs/mtslstmcmal_basins_finetuned_2606_173141")
eval_run(finetune_dir, period="test")

In [ ]:
# load test results of the base run
df_pretrained = pd.read_csv(run_dir / "test/model_epoch050/test_metrics.csv", dtype={'basin': str})
df_pretrained = df_pretrained.set_index("basin")
    
# load test results of the finetuned model
df_finetuned = pd.read_csv(finetune_dir / "test/model_epoch050/test_metrics.csv", dtype={'basin': str})
df_finetuned = df_finetuned.set_index("basin")
    
# extract basin performance
base_model_nse = df_pretrained.loc[df_pretrained.index == basin, "NSE_1D"].values[0]
finetune_nse = df_finetuned.loc[df_finetuned.index == basin, "NSE_1D"].values[0]
print(f"Basin {basin} base model performance: {base_model_nse:.3f}")
print(f"Performance after finetuning: {finetune_nse:.3f}")

In [ ]:
from neuralhydrology.evaluation.plots import uncertainty_plot, percentile_plot

# --- 2. Setup: Define run directory and load the config ---
# Make sure 'run_dir' is pointing to your correctly evaluated run folder
try:
    cfg = Config(run_dir / "config.yml")
except NameError:
    raise NameError("The 'run_dir' variable is not defined. Please define it before this cell.")

# --- 3. Configuration ---
basin_id = '01031500'       # The basin ID you are analyzing
freq = '1h'                 # Choose the frequency to plot: '1h' or '1D'
target_variable = cfg.target_variables[0] # Dynamically get target from config

# Define the variable names we need to extract
obs_var_name = f"{target_variable}_obs"

# The UncertaintyTester saves samples under the '_sim' key.
samples_var_name = f"{target_variable}_sim"


print(f"Generating uncertainty plot for basin {basin_id} at {freq} frequency...")

# --- 4. Data Extraction and Reshaping ---
try:
    xr_data = results[basin_id][freq]['xr']
    
    # Check if the crucial samples variable exists
    if samples_var_name not in xr_data:
        # Update the error message to reflect the new variable name
        raise KeyError(f"The samples variable '{samples_var_name}' was not found!")

    # Extract data as NumPy arrays
    obs_data_xr = xr_data[obs_var_name].values
    samples_data_xr = xr_data[samples_var_name].values
    
    # Reshape the data to the format expected by the plotting functions
    # The functions need (timesteps, features, samples)
    
    # Reshape observations: flatten date/time_step dimensions and add a 'feature' dimension
    y_obs_reshaped = obs_data_xr.reshape(-1, obs_data_xr.shape[-1])
    
    # Reshape samples: flatten date/time_step dimensions
    num_samples = samples_data_xr.shape[-1]
    y_samples_reshaped = samples_data_xr.reshape(-1, samples_data_xr.shape[-2], num_samples)

except KeyError as e:
    print(f"CRITICAL ERROR: {e}")
    print("\nSomething went wrong during data extraction.")
    print(f"Available variables in the results file are: {list(xr_data.keys())}")
    raise # Stop execution

# --- 5. Call the Official Plotting Function ---

# OPTION A: Use uncertainty_plot (plots reliability + a 400-step hydrograph zoom-in)
print("Using 'uncertainty_plot' (reliability + hydrograph zoom-in)...")
fig, ax = uncertainty_plot(y=y_obs_reshaped,
                           y_hat=y_samples_reshaped,
                           title=f"Uncertainty for Basin {basin_id} ({freq})")
plt.show()


# OPTION B: Use percentile_plot (plots the full hydrograph with uncertainty bands)
# This is a great alternative if you want to see the whole time series.
print("\nUsing 'percentile_plot' (full hydrograph)...")
fig2, ax2 = percentile_plot(y=y_obs_reshaped,
                            y_hat=y_samples_reshaped,
                            title=f"Full Timeseries Uncertainty for Basin {basin_id} ({freq})")
plt.show()

In [ ]:
# Make sure 'run_dir' is pointing to your correctly evaluated run folder
try:
    cfg = Config(run_dir / "config.yml")
except NameError:
    raise NameError("The 'run_dir' variable is not defined. Please define it before this cell.")

# --- 3. Configuration ---
basin_id = '01031500'       # The basin ID you are analyzing
target_variable = cfg.target_variables[0] # Dynamically get target from config

# Define the variable names to look for in the results file
obs_var_name = f"{target_variable}_obs"

# ######################################################################
# ################         THIS IS THE CRITICAL FIX         ##############
# The UncertaintyTester saves samples under the '_sim' key.
samples_var_name = f"{target_variable}_sim"
# ######################################################################
# ######################################################################


# --- 4. Loop through frequencies and run analysis ---
if basin_id not in results:
    raise ValueError(f"Basin {basin_id} not found in results. Available basins: {list(results.keys())}")

for frequency in cfg.use_frequencies: 
    print("======================================================================")
    print(f"      Running Probabilistic Analysis for Timescale: {frequency}      ")
    print("======================================================================")

    try:
        # Extract the xarray Dataset for the current frequency
        xr_data = results[basin_id][frequency]['xr']
        
        # Extract observations and simulation samples using the correct variable names
        obs_data_multidim = xr_data[obs_var_name].values
        sim_samples_multidim = xr_data[samples_var_name].values

        # --- Data Reshaping to get 1D (obs) and 2D (samples) arrays ---
        # Flatten all time dimensions into a single long timeseries
        obs_data_flat = obs_data_multidim.flatten()
        
        # Reshape samples to (total_timesteps, n_samples)
        num_samples = sim_samples_multidim.shape[-1]
        sim_samples_reshaped = sim_samples_multidim.reshape(-1, num_samples)
            
        # --- Handle NaNs ---
        valid_indices = ~np.isnan(obs_data_flat)
        if not np.all(valid_indices):
            print(f"Warning: Found and removed {np.sum(~valid_indices)} NaN values in observations.")
            obs_data = obs_data_flat[valid_indices]
            sim_samples = sim_samples_reshaped[valid_indices, :]
        else:
            obs_data = obs_data_flat
            sim_samples = sim_samples_reshaped

        # --- Run the analysis ---
        probabilistic_results = run_probabilistic_evaluation(obs=obs_data, sim_samples=sim_samples)

    except KeyError as e:
        print(f"\nCould not run analysis for frequency '{frequency}'.")
        # Update the error message to be more informative
        print(f"A required data variable was not found. Tried to find '{obs_var_name}' and '{samples_var_name}'.")
        print(f"Available variables are: {list(xr_data.keys())}\n")
    except Exception as e:
        print(f"\nAn unexpected error occurred for frequency '{frequency}': {e}\n")